# Query-Likelihood Language Models and Smoothing

This notebook is the *narrative* pillar: it imports the canonical reference implementation in `query_likelihood_language_models.py` — which **owns every number** — and walks the topic section by section. Each mathematical claim is an `assert` in the `.py`; here we run those checks and print the numbers the page and the interactive viz mirror.

Run end to end with:

```bash
uv run --with numpy --with jupyter \
    jupyter execute notebooks/query-likelihood-language-models/01_query_likelihood_language_models.ipynb
```

In [ ]:
import sys, pathlib
_cands = [pathlib.Path.cwd(),
          pathlib.Path.cwd() / "notebooks" / "query-likelihood-language-models",
          pathlib.Path("notebooks/query-likelihood-language-models")]
for _d in _cands:
    if (_d / "query_likelihood_language_models.py").exists():
        sys.path.insert(0, str(_d)); break
import query_likelihood_language_models as ql
index = ql.build_inverted_index(ql._FINANCE_CORPUS)
print("loaded reference implementation; query =", ql._QUERY)

## 1. Documents and the query as language models

We model each document $d$ as a **multinomial unigram language model** $M_d$ over the vocabulary, and rank by the probability that $M_d$ generated the query: $P(q \mid M_d) = \prod_{t \in q} P(t \mid M_d)^{c(t,q)}$. The **collection model** $P(t \mid C) = \mathrm{cf}_t / |C|$ is the background distribution used by smoothing. We reuse the finance corpus from BM25 and the vector space model verbatim, so this is the *same* set of documents scored by a new model.

In [ ]:
ql.collection_table()
for d, L in zip(index.doc_ids, index.doc_len.astype(int)):
    print(f'  {d:<16} |d|={L:>3} tokens')

## 2. The query-likelihood score and the zero-frequency catastrophe

The maximum-likelihood estimate $P_{\mathrm{ml}}(t \mid d) = \mathrm{tf}_{t,d}/|d|$ already divides by length, so query likelihood does **not** suffer the length hijack that captured the raw tf-idf dot product. But any document missing even one query term gets $P_{\mathrm{ml}} = 0$, so $\log P(q \mid d) = -\infty$ — the **zero-frequency catastrophe**. On this corpus it kills exactly the three documents missing *interest*, *rate*, or *exposure*.

In [ ]:
ql.test_zero_frequency_catastrophe()

## 3. Jelinek–Mercer and Dirichlet smoothing

Smoothing rescues unseen terms by mixing in the collection model. **Jelinek–Mercer** is a fixed linear interpolation $P_\lambda(t\mid d) = (1-\lambda)P_{\mathrm{ml}}(t\mid d) + \lambda P(t\mid C)$; **Dirichlet** is $P_\mu(t\mid d) = (\mathrm{tf}_{t,d} + \mu P(t\mid C))/(|d| + \mu)$. Both produce valid probability distributions over the vocabulary.

In [ ]:
ql.validate_normalization()

## 4. Dirichlet smoothing is a Bayesian posterior mean (Theorem 1)

Place a conjugate **Dirichlet prior** with concentration $\alpha_t = \mu P(t\mid C)$ on the document's multinomial parameter. Observing the document's counts updates the posterior to $\mathrm{Dirichlet}(\alpha_t + \mathrm{tf}_t)$, whose mean is exactly $(\mathrm{tf}_t + \mu P(t\mid C))/(|d| + \mu)$. Dirichlet smoothing *is* Bayesian estimation under a collection-model prior with strength $\mu$.

In [ ]:
ql.test_dirichlet_is_posterior_mean()

## 5. The KL-divergence view (Theorem 2)

Let $\theta_q(t) = c(t,q)/|q|$ be the empirical query model. Ranking by the negative KL divergence $-\mathrm{KL}(\theta_q \,\|\, \theta_d)$ is **rank-equivalent** to the query likelihood, because $-\mathrm{KL} = H(\theta_q) + \sum_t \theta_q(t)\log P(t\mid d)$ and the query entropy $H(\theta_q)$ is constant across documents. This is the view that pseudo-relevance feedback (RM3) generalizes by replacing $\theta_q$ with a richer relevance model.

In [ ]:
ql.test_kl_rank_equivalence()

## 6. Smoothing plays an IDF-like role (Theorem 3)

The Jelinek–Mercer log score splits into a document-dependent matched-term sum plus a document-independent constant:
$$\log P(q\mid d) = \sum_{t\in q,\,\mathrm{tf}>0} c(t,q)\,\log\!\Big(1 + \tfrac{1-\lambda}{\lambda}\,\tfrac{P_{\mathrm{ml}}(t\mid d)}{P(t\mid C)}\Big) + |q|\log\lambda + \sum_{t\in q}c(t,q)\log P(t\mid C).$$
Only the first sum changes the ranking, and inside it the per-term weight rises as the collection probability $P(t\mid C)$ falls — an IDF-like effect that *emerges* from smoothing rather than being put in by hand.

In [ ]:
ql.test_jm_decomposition_idf_effect()

## 7. Length adaptivity and the contrast with the length hijack

Dirichlet's effective smoothing weight $\mu/(|d|+\mu)$ shrinks as $|d|$ grows, so short documents are smoothed more and the score carries an explicit length penalty $|q|\log(\mu/(|d|+\mu))$ — the length-normalization role of smoothing. The payoff: on the very corpus where the raw tf-idf dot product crowned the padded transcript, query likelihood (with either smoothing) puts the concise on-point filing first.

In [ ]:
ql.test_dirichlet_length_adaptivity()
ql.test_ql_resists_length_hijack()
ql.smoothing_demo()

## 8. The numbers the viz mirrors

`QueryLikelihoodSmoothingLab.tsx` recomputes the document models from the query-term frequencies, but the collection model $P(t\mid C)$ and the document lengths depend on the whole corpus (including filler) and are mirrored from here to the decimal.

In [ ]:
ql.viz_constants()

---

**Three pillars, one set of numbers.** The rigorous math (the page), the interactive smoothing laboratory, and this notebook all read from `query_likelihood_language_models.py`. Change a number there and the viz and the page must change with it.